# JSON documents, step by step

A hands-on lab. Run one cell at a time, read what happened, then move on. You will query JSONB specs with containment, path extraction, and GIN indexes.

Each step shows the **plain SQL** too, so you can copy it and run it in psql, the Lakebase SQL editor, or any tool. The Python just runs that SQL against your branch.

**Before you start:** finish `00-setup/` so `LB_PROFILE`, `LB_PROJECT`, `LB_BRANCH`, `LB_SCHEMA` are set and the bank is loaded on your branch.

## Setup, connect to your branch

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'common', 'scripts'))
from lakebase import connect, run_sql

SCHEMA = os.environ.get('LB_SCHEMA', 'bank')
print('connected; schema:', SCHEMA)

## Step 1, inspect JSONB specs on instruments

Each instrument has a `spec` column (JSONB). The contents vary: one instrument might have `{"ticker": "AAPL", "sector": "tech", "beta": 1.2}`, another might have `{"tenor": "10Y", "coupon": 4.0, "rating": "AAA"}`. No schema migration needed when a new instrument type arrives with new fields. The SQL:

```sql
SELECT instrument_id, name, spec FROM bank.instrument LIMIT 3
```

In [ ]:
sql = "SELECT instrument_id, name, spec FROM bank.instrument LIMIT 3"
rows = run_sql(sql)
print("Sample instrument specs (JSONB, varying fields):")
for row in rows:
    print(f"  {row[0]}: {row[1]}")
    print(f"    spec: {row[2]}")

## Step 2, use containment operator (@>) to find instruments matching a spec

The `@>` operator asks "does the JSON document on the left contain the JSON sub-document on the right?" This is the workhorse for spec matching. Find instruments that match a spec. The SQL:

```sql
SELECT instrument_id, name, spec
  FROM bank.instrument
 WHERE spec @> '{"sector": "tech"}'
 LIMIT 5
```

In [ ]:
sql = """
SELECT instrument_id, name, spec
  FROM bank.instrument
 WHERE spec @> '{"sector": "tech"}'
 LIMIT 5
"""
rows = run_sql(sql)
print("Instruments with sector='tech' (using @> containment):")
for row in rows:
    print(f"  {row[0]}: {row[1]}")
    print(f"    spec: {row[2]}")

## Step 3, use path extraction (#>>) and key exists (?) operators

The `#>>` operator extracts a value from a path in the JSON. Use it to extract a field for filtering or display. The `?` operator checks if a key exists. Together, they let you filter and display JSON fields like columns. The SQL:

```sql
SELECT instrument_id, name, spec #>> '{sector}' as sector
  FROM bank.instrument
 WHERE spec ? 'sector'
 LIMIT 5
```

In [ ]:
sql = """
SELECT instrument_id, name, spec #>> '{sector}' as sector
  FROM bank.instrument
 WHERE spec ? 'sector'
 LIMIT 5
"""
rows = run_sql(sql)
print("Extracted sector field from JSONB spec:")
for row in rows:
    print(f"  {row[0]}: sector={row[2]}")

## Step 4, create a GIN index on JSONB

JSONB queries are fast even without an index. With a GIN (Generalized Inverted Index) on the JSONB column, they are faster. The SQL:

```sql
CREATE INDEX IF NOT EXISTS idx_instrument_spec_gin ON bank.instrument
  USING gin(spec jsonb_path_ops)
```

In [ ]:
sql = "CREATE INDEX IF NOT EXISTS idx_instrument_spec_gin ON bank.instrument USING gin(spec jsonb_path_ops)"
with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)
print("GIN index created on bank.instrument(spec)")
print("(Queries will use the index if beneficial)")

## Step 5, realistic screening query: equities in tech sector

Find instruments that are equities in the tech sector. The SQL:

```sql
SELECT instrument_id, name, spec
  FROM bank.instrument
 WHERE spec @> '{"sector": "tech"}'
   AND kind = 'equity'
 LIMIT 10
```

In [ ]:
sql = """
SELECT instrument_id, name, spec
  FROM bank.instrument
 WHERE spec @> '{"sector": "tech"}'
   AND kind = 'equity'
 LIMIT 10
"""
rows = run_sql(sql)
print("Equities in tech sector:")
for row in rows:
    print(f"  {row[0]}: {row[1]}")
    if row[2]:
        print(f"    {row[2]}")

## What you learned

You mastered JSONB: the containment operator (@>) for semantic matching, path extraction (#>>) and key checks (?) for drill-down, and GIN indexes for speed. Postgres gives you flexibility of schema-less storage with the power of a relational query engine.

Next: **Geospatial queries** use PostGIS to find branches, custodians, and counterparties within a distance, nearest neighbors, and spatial indexes.